In [1]:
# ===== Notebook 2 setup: restore Step 5 outputs into /kaggle/working =====
import glob, os, shutil
src = glob.glob("/kaggle/input/**/feats", recursive=True)[0]
shutil.copytree(src, "/kaggle/working/feats", dirs_exist_ok=True)
shutil.copytree(src.replace("feats", "figures"), "/kaggle/working/figures", dirs_exist_ok=True)
print("feats:", len(os.listdir("/kaggle/working/feats")), "| figures:", len(os.listdir("/kaggle/working/figures")))

feats: 13 | figures: 39


In [2]:
# ===== Step 6.1: write train_student.py + timing run on PlantDoc P2 =====
import os, subprocess, time, json, pandas as pd
os.makedirs("/kaggle/working/students", exist_ok=True)
SCRIPT = r'''
import argparse, os, time, json, numpy as np, pandas as pd, torch, torch.nn as nn, timm
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score
p = argparse.ArgumentParser()
p.add_argument("--data", required=True); p.add_argument("--proto", required=True)
p.add_argument("--model", required=True); p.add_argument("--seed", type=int, default=0)
p.add_argument("--epochs", type=int, default=40); p.add_argument("--bs", type=int, default=64)
p.add_argument("--lr", type=float, default=5e-4); p.add_argument("--wd", type=float, default=0.05)
p.add_argument("--feat", default="/kaggle/working/feats"); p.add_argument("--out", default="/kaggle/working/students")
a = p.parse_args()
PROC = {"pd": "/kaggle/input/datasets/mashfii/plantdoc-clean-thesis/proc",
        "pw": "/kaggle/input/datasets/mashfii/plantwild-clean-thesis/proc"}
torch.manual_seed(a.seed); np.random.seed(a.seed); torch.backends.cudnn.benchmark = True
df = pd.read_csv(f"{a.feat}/{a.data}_manifest.csv", low_memory=False)
df["img"] = df.proc_path.map(lambda s: os.path.join(PROC[a.data], str(s)[str(s).find("/proc/")+6:]))
K = int(df.class_id.max()) + 1
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
tf_tr = T.Compose([T.RandomResizedCrop(224, scale=(0.5, 1.0), interpolation=T.InterpolationMode.BICUBIC),
                   T.RandomHorizontalFlip(), T.ColorJitter(0.2, 0.2, 0.2), T.ToTensor(), T.Normalize(MEAN, STD)])
tf_ev = T.Compose([T.Resize(224, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224),
                   T.ToTensor(), T.Normalize(MEAN, STD)])
class DS(Dataset):
    def __init__(s, d, tf): s.p, s.y, s.tf = d.img.values, d.class_id.values, tf
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return s.tf(Image.open(s.p[i]).convert("RGB")), s.y[i]
split = {r: df[df[a.proto] == r] for r in ["train", "val", "test"]}
dl_tr = DataLoader(DS(split["train"], tf_tr), a.bs, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
dl = {r: DataLoader(DS(split[r], tf_ev), 128, num_workers=4, pin_memory=True) for r in ["val", "test"]}
model = timm.create_model(a.model, pretrained=True, num_classes=K).cuda()
n_params = sum(p.numel() for p in model.parameters())
opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.wd)
steps = a.epochs * len(dl_tr); warm = len(dl_tr)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s+1)/warm if s < warm else 0.5*(1+np.cos(np.pi*(s-warm)/(steps-warm))))
crit = nn.CrossEntropyLoss(label_smoothing=0.1); scaler = torch.amp.GradScaler()
@torch.no_grad()
def predict(loader):
    model.eval(); L, Y = [], []
    for x, y in loader:
        with torch.amp.autocast("cuda"): L.append(model(x.cuda(non_blocking=True)).float().cpu())
        Y.append(y)
    return torch.cat(L).numpy(), torch.cat(Y).numpy()
curve, best, t0 = [], (-1, None, -1), time.time()
for ep in range(a.epochs):
    model.train(); tl, n = 0.0, 0
    for x, y in dl_tr:
        x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
        with torch.amp.autocast("cuda"): loss = crit(model(x), y)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tl += loss.item() * len(y); n += len(y)
    lv, yv = predict(dl["val"]); f1v = f1_score(yv, lv.argmax(1), average="macro"); accv = accuracy_score(yv, lv.argmax(1))
    curve.append(dict(epoch=ep+1, train_loss=tl/n, val_acc=accv, val_f1=f1v, time=time.time()-t0))
    if f1v > best[0]: best = (f1v, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, ep+1)
model.load_state_dict(best[1])
lv, yv = predict(dl["val"]); lt, yt = predict(dl["test"])
tag = f"{a.data}_{a.proto.replace('_role','')}_{a.model.split('.')[0]}_s{a.seed}"
res = dict(tag=tag, data=a.data, proto=a.proto, model=a.model, seed=a.seed, params_M=n_params/1e6,
           best_epoch=best[2], val_f1=f1_score(yv, lv.argmax(1), average="macro"),
           test_acc=accuracy_score(yt, lt.argmax(1)), test_f1=f1_score(yt, lt.argmax(1), average="macro"),
           minutes=(time.time()-t0)/60, n_train=len(split["train"]))
os.makedirs(f"{a.out}/logits", exist_ok=True); os.makedirs(f"{a.out}/curves", exist_ok=True)
np.save(f"{a.out}/logits/{tag}_test.npy", lt.astype(np.float16)); np.save(f"{a.out}/logits/{tag}_val.npy", lv.astype(np.float16))
split["test"][["uid"]].to_csv(f"{a.out}/logits/{tag}_test_uids.csv", index=False)
pd.DataFrame(curve).to_csv(f"{a.out}/curves/{tag}.csv", index=False)
with open(f"{a.out}/results.jsonl", "a") as f: f.write(json.dumps(res) + "\n")
print("RESULT", json.dumps(res))
'''
open("/kaggle/working/train_student.py", "w").write(SCRIPT)

# --- timing run: two jobs in parallel, one per GPU ---
jobs = [("0", "mobilenetv3_large_100.ra_in1k"), ("1", "efficientnet_b0.ra_in1k")]
procs, t0 = [], time.time()
for gpu, m in jobs:
    log = open(f"/kaggle/working/students/log_{m.split('.')[0]}.txt", "w")
    procs.append(subprocess.Popen(["python", "/kaggle/working/train_student.py", "--data", "pd", "--proto", "p2_role",
                                   "--model", m, "--seed", "0", "--epochs", "40"],
                                  env={**os.environ, "CUDA_VISIBLE_DEVICES": gpu}, stdout=log, stderr=subprocess.STDOUT))
for p in procs: p.wait()
print(f"wall time: {(time.time()-t0)/60:.1f} min")
for gpu, m in jobs:
    txt = open(f"/kaggle/working/students/log_{m.split('.')[0]}.txt").read()
    print(f"\n--- {m} ---"); print("\n".join([l for l in txt.splitlines() if "RESULT" in l or "Error" in l or "error" in l][-5:]) or txt[-1500:])
    c = pd.read_csv(f"/kaggle/working/students/curves/pd_p2_{m.split('.')[0]}_s0.csv")
    print(c.iloc[[0, 4, 9, 19, 29, 39]].round(4).to_string(index=False))
print("\nStep 6.1 DONE.")

wall time: 11.3 min

--- mobilenetv3_large_100.ra_in1k ---
RESULT {"tag": "pd_p2_mobilenetv3_large_100_s0", "data": "pd", "proto": "p2_role", "model": "mobilenetv3_large_100.ra_in1k", "seed": 0, "params_M": 4.236619, "best_epoch": 33, "val_f1": 0.6599050566820767, "test_acc": 0.6, "test_f1": 0.5662197289279595, "minutes": 10.688988796869914, "n_train": 1872}
 epoch  train_loss  val_acc  val_f1     time
     1      3.2123   0.4295  0.3763  77.6709
     5      0.8704   0.5833  0.5573 131.8999
    10      0.7471   0.6250  0.5956 204.0857
    20      0.6925   0.6538  0.6260 344.2514
    30      0.6781   0.6442  0.6197 483.0697
    40      0.6744   0.6538  0.6294 627.9533

--- efficientnet_b0.ra_in1k ---
RESULT {"tag": "pd_p2_efficientnet_b0_s0", "data": "pd", "proto": "p2_role", "model": "efficientnet_b0.ra_in1k", "seed": 0, "params_M": 4.042135, "best_epoch": 12, "val_f1": 0.6510790327843244, "test_acc": 0.6181818181818182, "test_f1": 0.590769321762726, "minutes": 11.002942391236623, "n_t

In [3]:
# ===== Step 6.2: uint8 image cache + cached training script + LR sweep (PD P2) =====
import os, time, subprocess, json, numpy as np, pandas as pd
from PIL import Image
from multiprocessing import Pool
FEAT, CACHE = "/kaggle/working/feats", "/kaggle/working/cache"; os.makedirs(CACHE, exist_ok=True)
PROC = {"pd": "/kaggle/input/datasets/mashfii/plantdoc-clean-thesis/proc",
        "pw": "/kaggle/input/datasets/mashfii/plantwild-clean-thesis/proc"}
S = 256
def load256(path):
    im = Image.open(path).convert("RGB"); w, h = im.size; r = S / min(w, h)
    im = im.resize((max(S, round(w*r)), max(S, round(h*r))), Image.BICUBIC); w, h = im.size
    l, t = (w - S)//2, (h - S)//2
    return np.asarray(im.crop((l, t, l+S, t+S)), dtype=np.uint8)
for tag in ["pd", "pw"]:
    out = f"{CACHE}/{tag}_256.npy"
    if os.path.exists(out): print("exists", out); continue
    df = pd.read_csv(f"{FEAT}/{tag}_manifest.csv", low_memory=False)
    paths = [os.path.join(PROC[tag], str(s)[str(s).find("/proc/")+6:]) for s in df.proc_path]
    arr = np.lib.format.open_memmap(out, mode="w+", dtype=np.uint8, shape=(len(paths), S, S, 3))
    t = time.time()
    with Pool(4) as pool:
        for i, a in enumerate(pool.imap(load256, paths, chunksize=64)): arr[i] = a
    arr.flush(); del arr
    print(f"{tag}: {len(paths)} imgs cached in {time.time()-t:.0f}s -> {os.path.getsize(out)/1e9:.2f} GB")

SCRIPT = r'''
import argparse, os, time, json, numpy as np, pandas as pd, torch, torch.nn as nn, timm
from torchvision.transforms import v2 as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score
p = argparse.ArgumentParser()
p.add_argument("--data", required=True); p.add_argument("--proto", required=True)
p.add_argument("--model", required=True); p.add_argument("--seed", type=int, default=0)
p.add_argument("--epochs", type=int, default=30); p.add_argument("--bs", type=int, default=64)
p.add_argument("--lr", type=float, default=5e-4); p.add_argument("--wd", type=float, default=0.05)
p.add_argument("--feat", default="/kaggle/working/feats"); p.add_argument("--cache", default="/kaggle/working/cache")
p.add_argument("--out", default="/kaggle/working/students"); p.add_argument("--tag_extra", default="")
a = p.parse_args()
torch.manual_seed(a.seed); np.random.seed(a.seed); torch.backends.cudnn.benchmark = True
df = pd.read_csv(f"{a.feat}/{a.data}_manifest.csv", low_memory=False)
K = int(df.class_id.max()) + 1
ARR = f"{a.cache}/{a.data}_256.npy"
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
tf_tr = T.Compose([T.RandomResizedCrop(224, scale=(0.5, 1.0), antialias=True), T.RandomHorizontalFlip(),
                   T.ColorJitter(0.2, 0.2, 0.2), T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)])
tf_ev = T.Compose([T.Resize(224, antialias=True), T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)])
class DS(Dataset):
    def __init__(s, idx, y, tf): s.idx, s.y, s.tf, s.arr = idx, y, tf, None
    def __len__(s): return len(s.idx)
    def __getitem__(s, i):
        if s.arr is None: s.arr = np.load(ARR, mmap_mode="r")
        x = torch.from_numpy(np.ascontiguousarray(s.arr[s.idx[i]])).permute(2, 0, 1)
        return s.tf(x), s.y[i]
split = {r: np.where(df[a.proto].values == r)[0] for r in ["train", "val", "test"]}
y_all = df.class_id.values
mk = lambda r, tf: DS(split[r], y_all[split[r]], tf)
dl_tr = DataLoader(mk("train", tf_tr), a.bs, shuffle=True, num_workers=3, pin_memory=True, drop_last=True, persistent_workers=True)
dl = {r: DataLoader(mk(r, tf_ev), 128, num_workers=2, pin_memory=True) for r in ["val", "test"]}
model = timm.create_model(a.model, pretrained=True, num_classes=K).cuda()
n_params = sum(p.numel() for p in model.parameters())
opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.wd)
steps = a.epochs * len(dl_tr); warm = len(dl_tr)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s+1)/warm if s < warm else 0.5*(1+np.cos(np.pi*(s-warm)/max(1, steps-warm))))
crit = nn.CrossEntropyLoss(label_smoothing=0.1); scaler = torch.amp.GradScaler()
@torch.no_grad()
def predict(loader):
    model.eval(); L, Y = [], []
    for x, y in loader:
        with torch.amp.autocast("cuda"): L.append(model(x.cuda(non_blocking=True)).float().cpu())
        Y.append(y)
    return torch.cat(L).numpy(), torch.cat(Y).numpy()
curve, best, t0 = [], (-1, None, -1), time.time()
for ep in range(a.epochs):
    model.train(); tl, n = 0.0, 0
    for x, y in dl_tr:
        x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
        with torch.amp.autocast("cuda"): loss = crit(model(x), y)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tl += loss.item() * len(y); n += len(y)
    lv, yv = predict(dl["val"]); f1v = f1_score(yv, lv.argmax(1), average="macro")
    curve.append(dict(epoch=ep+1, train_loss=tl/n, val_acc=accuracy_score(yv, lv.argmax(1)), val_f1=f1v, time=time.time()-t0))
    if f1v > best[0]: best = (f1v, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, ep+1)
model.load_state_dict(best[1])
lv, yv = predict(dl["val"]); lt, yt = predict(dl["test"])
tag = f"{a.data}_{a.proto.replace('_role','')}_{a.model.split('.')[0]}_s{a.seed}_lr{a.lr:g}{a.tag_extra}"
res = dict(tag=tag, data=a.data, proto=a.proto, model=a.model, seed=a.seed, lr=a.lr, epochs=a.epochs,
           params_M=n_params/1e6, best_epoch=best[2], val_f1=f1_score(yv, lv.argmax(1), average="macro"),
           test_acc=accuracy_score(yt, lt.argmax(1)), test_f1=f1_score(yt, lt.argmax(1), average="macro"),
           minutes=(time.time()-t0)/60, n_train=len(split["train"]))
for d in ["logits", "curves", "ckpt"]: os.makedirs(f"{a.out}/{d}", exist_ok=True)
np.save(f"{a.out}/logits/{tag}_test.npy", lt.astype(np.float16)); np.save(f"{a.out}/logits/{tag}_val.npy", lv.astype(np.float16))
pd.DataFrame({"row": split["test"], "uid": df.uid.values[split["test"]]}).to_csv(f"{a.out}/logits/{tag}_test_uids.csv", index=False)
pd.DataFrame(curve).to_csv(f"{a.out}/curves/{tag}.csv", index=False)
torch.save(best[1], f"{a.out}/ckpt/{tag}.pt")
with open(f"{a.out}/results.jsonl", "a") as f: f.write(json.dumps(res) + "\n")
print("RESULT", json.dumps(res))
'''
open("/kaggle/working/train_student.py", "w").write(SCRIPT)

def run_pair(jobs):
    procs = []
    for gpu, args in enumerate(jobs):
        tag = "_".join(a for a in args if not a.startswith("--"))[:80]
        log = open(f"/kaggle/working/students/log_{tag}.txt", "w")
        procs.append((subprocess.Popen(["python", "/kaggle/working/train_student.py", *args],
                      env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}, stdout=log, stderr=subprocess.STDOUT), tag))
    for p, tag in procs:
        p.wait()
        if p.returncode != 0: print("FAILED", tag, open(f"/kaggle/working/students/log_{tag}.txt").read()[-1500:])

MODELS = ["mobilenetv3_large_100.ra_in1k", "efficientnet_b0.ra_in1k"]
t0 = time.time()
for lr in ["2e-4", "5e-4", "1e-3"]:
    run_pair([["--data", "pd", "--proto", "p2_role", "--model", m, "--seed", "0", "--lr", lr, "--epochs", "30"] for m in MODELS])
    print(f"lr={lr} done  ({(time.time()-t0)/60:.1f} min)")

R = pd.read_json("/kaggle/working/students/results.jsonl", lines=True)
R = R[(R.data == "pd") & (R.proto == "p2_role") & (R.epochs == 30)]
print("\n=== LR SWEEP (PlantDoc P2; select on val_f1) ===")
print(R[["model", "lr", "best_epoch", "val_f1", "test_f1", "minutes"]].round(4).sort_values(["model", "lr"]).to_string(index=False))
print("\nStep 6.2 DONE.")

pd: 2576 imgs cached in 9s -> 0.51 GB
pw: 18542 imgs cached in 73s -> 3.65 GB
lr=2e-4 done  (6.0 min)
lr=5e-4 done  (12.1 min)
lr=1e-3 done  (18.1 min)

=== LR SWEEP (PlantDoc P2; select on val_f1) ===
                        model     lr  best_epoch  val_f1  test_f1  minutes
      efficientnet_b0.ra_in1k 0.0002          25  0.5998   0.5656   5.5766
      efficientnet_b0.ra_in1k 0.0005          14  0.6384   0.6223   5.6678
      efficientnet_b0.ra_in1k 0.0010          27  0.6890   0.5749   5.6008
mobilenetv3_large_100.ra_in1k 0.0002          23  0.6041   0.5634   4.8996
mobilenetv3_large_100.ra_in1k 0.0005          25  0.6217   0.5795   5.0089
mobilenetv3_large_100.ra_in1k 0.0010          22  0.6378   0.5990   4.9496

Step 6.2 DONE.


In [4]:
# ===== Step 6.3: GPU-augmentation trainer + PlantDoc P3 baseline grid =====
import os, time, subprocess, json, pandas as pd, numpy as np
SCRIPT = r'''
import argparse, os, time, json, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, timm
from torchvision.ops import roi_align
from sklearn.metrics import f1_score, accuracy_score
p = argparse.ArgumentParser()
p.add_argument("--data", required=True); p.add_argument("--proto", required=True)
p.add_argument("--model", required=True); p.add_argument("--seed", type=int, default=0)
p.add_argument("--epochs", type=int, default=30); p.add_argument("--bs", type=int, default=64)
p.add_argument("--lr", type=float, default=1e-3); p.add_argument("--wd", type=float, default=0.05)
p.add_argument("--feat", default="/kaggle/working/feats"); p.add_argument("--cache", default="/kaggle/working/cache")
p.add_argument("--out", default="/kaggle/working/students"); p.add_argument("--tag_extra", default="")
a = p.parse_args()
torch.manual_seed(a.seed); np.random.seed(a.seed); torch.backends.cudnn.benchmark = True
dev = "cuda"
df = pd.read_csv(f"{a.feat}/{a.data}_manifest.csv", low_memory=False)
K = int(df.class_id.max()) + 1
X = torch.from_numpy(np.load(f"{a.cache}/{a.data}_256.npy")).to(dev)          # N,256,256,3 uint8 on GPU
Y = torch.tensor(df.class_id.values, device=dev)
split = {r: torch.tensor(np.where(df[a.proto].values == r)[0], device=dev) for r in ["train", "val", "test"]}
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
S = 256
def to_float(idx): return X[idx].permute(0, 3, 1, 2).float() / 255.
def augment(x):
    B = x.shape[0]
    area = S * S * torch.empty(B, device=dev).uniform_(0.5, 1.0)
    logr = torch.empty(B, device=dev).uniform_(np.log(3/4), np.log(4/3)); r = torch.exp(logr)
    w = torch.sqrt(area * r).clamp(max=S); h = torch.sqrt(area / r).clamp(max=S)
    x0 = torch.rand(B, device=dev) * (S - w); y0 = torch.rand(B, device=dev) * (S - h)
    boxes = torch.stack([torch.arange(B, device=dev).float(), x0, y0, x0 + w, y0 + h], 1)
    x = roi_align(x, boxes, output_size=224, spatial_scale=1.0, sampling_ratio=2, aligned=True)
    flip = torch.rand(B, device=dev) < 0.5; x[flip] = x[flip].flip(-1)
    u = lambda: torch.empty(B, 1, 1, 1, device=dev).uniform_(0.8, 1.2)
    x = x * u()                                                                    # brightness
    m = x.mean((1, 2, 3), keepdim=True); x = (x - m) * u() + m                     # contrast
    g = (0.299*x[:, 0:1] + 0.587*x[:, 1:2] + 0.114*x[:, 2:3]); x = (x - g) * u() + g  # saturation
    return ((x.clamp(0, 1) - MEAN) / STD)
def prep_eval(x): return (F.interpolate(x, size=224, mode="bicubic", antialias=True, align_corners=False).clamp(0, 1) - MEAN) / STD
model = timm.create_model(a.model, pretrained=True, num_classes=K).to(dev).to(memory_format=torch.channels_last)
n_params = sum(p.numel() for p in model.parameters())
opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.wd)
ntr = len(split["train"]); spe = ntr // a.bs; steps = a.epochs * spe; warm = spe
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s+1)/warm if s < warm else 0.5*(1+np.cos(np.pi*(s-warm)/max(1, steps-warm))))
crit = nn.CrossEntropyLoss(label_smoothing=0.1); scaler = torch.amp.GradScaler()
@torch.no_grad()
def predict(r):
    model.eval(); L = []
    for i in range(0, len(split[r]), 256):
        idx = split[r][i:i+256]
        with torch.amp.autocast("cuda"): L.append(model(prep_eval(to_float(idx)).contiguous(memory_format=torch.channels_last)).float())
    return torch.cat(L).cpu().numpy(), Y[split[r]].cpu().numpy()
curve, best, t0 = [], (-1, None, -1), time.time()
for ep in range(a.epochs):
    model.train(); tl = 0.0; perm = split["train"][torch.randperm(ntr, device=dev)]
    for s in range(spe):
        idx = perm[s*a.bs:(s+1)*a.bs]
        x = augment(to_float(idx)).contiguous(memory_format=torch.channels_last); y = Y[idx]
        with torch.amp.autocast("cuda"): loss = crit(model(x), y)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tl += loss.item()
    lv, yv = predict("val"); f1v = f1_score(yv, lv.argmax(1), average="macro")
    curve.append(dict(epoch=ep+1, train_loss=tl/spe, val_acc=accuracy_score(yv, lv.argmax(1)), val_f1=f1v, time=time.time()-t0))
    if f1v > best[0]: best = (f1v, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, ep+1)
model.load_state_dict(best[1])
lv, yv = predict("val"); lt, yt = predict("test")
tag = f"{a.data}_{a.proto.replace('_role','')}_{a.model.split('.')[0]}_s{a.seed}_lr{a.lr:g}{a.tag_extra}"
res = dict(tag=tag, data=a.data, proto=a.proto, model=a.model, seed=a.seed, lr=a.lr, epochs=a.epochs, pipeline="gpu_aug",
           params_M=n_params/1e6, best_epoch=best[2], val_f1=f1_score(yv, lv.argmax(1), average="macro"),
           test_acc=accuracy_score(yt, lt.argmax(1)), test_f1=f1_score(yt, lt.argmax(1), average="macro"),
           minutes=(time.time()-t0)/60, n_train=ntr)
for d in ["logits", "curves", "ckpt"]: os.makedirs(f"{a.out}/{d}", exist_ok=True)
te = split["test"].cpu().numpy()
np.save(f"{a.out}/logits/{tag}_test.npy", lt.astype(np.float16)); np.save(f"{a.out}/logits/{tag}_val.npy", lv.astype(np.float16))
pd.DataFrame({"row": te, "uid": df.uid.values[te]}).to_csv(f"{a.out}/logits/{tag}_test_uids.csv", index=False)
pd.DataFrame(curve).to_csv(f"{a.out}/curves/{tag}.csv", index=False)
torch.save(best[1], f"{a.out}/ckpt/{tag}.pt")
with open(f"{a.out}/results.jsonl", "a") as f: f.write(json.dumps(res) + "\n")
print("RESULT", json.dumps(res))
'''
open("/kaggle/working/train_student.py", "w").write(SCRIPT)

def run_pair(jobs):
    procs = []
    for gpu, args in enumerate(jobs):
        tag = "_".join(x for x in args if not x.startswith("--"))[:90]
        log = open(f"/kaggle/working/students/log_{tag}.txt", "w")
        procs.append((subprocess.Popen(["python", "/kaggle/working/train_student.py", *args],
                      env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}, stdout=log, stderr=subprocess.STDOUT), tag))
    for p, tag in procs:
        p.wait()
        if p.returncode != 0: print("FAILED", tag, open(f"/kaggle/working/students/log_{tag}.txt").read()[-2000:])

MODELS = ["mobilenetv3_large_100.ra_in1k", "efficientnet_b0.ra_in1k",
          "mobilenetv4_conv_small.e2400_r224_in1k", "resnet50.a1_in1k"]
jobs = [["--data", "pd", "--proto", f"p3_f{f}_role", "--model", m, "--seed", "0", "--lr", "1e-3", "--epochs", "30"]
        for m in MODELS for f in range(5)]
t0 = time.time()
for i in range(0, len(jobs), 2):
    run_pair(jobs[i:i+2])
    print(f"pair {i//2+1}/{len(jobs)//2} done: {jobs[i][5].split('.')[0]} {jobs[i][3]} & {jobs[i+1][3]}  "
          f"({(time.time()-t0)/60:.1f} min)")

R = pd.read_json("/kaggle/working/students/results.jsonl", lines=True)
R = R[(R.data == "pd") & R.proto.str.startswith("p3_") & (R.get("pipeline") == "gpu_aug")]
print("\n=== PlantDoc P3 per fold ===")
print(R[["model", "proto", "best_epoch", "val_f1", "test_f1", "minutes"]].round(4).to_string(index=False))
T = R.groupby("model").agg(params_M=("params_M", "first"), test_f1=("test_f1", "mean"), f1_sd=("test_f1", "std"),
                           test_acc=("test_acc", "mean"), min_per_run=("minutes", "mean")).round(4).sort_values("params_M")
print("\n=== PlantDoc P3 summary (mean ± SD over 5 folds) ===\n", T.to_string())
os.makedirs("/kaggle/working/figures", exist_ok=True); T.to_csv("/kaggle/working/figures/table_students_pd_p3_baseline.csv")
print("\nStep 6.3 DONE.")


pair 1/10 done: mobilenetv3_large_100 p3_f0_role & p3_f1_role  (2.9 min)
pair 2/10 done: mobilenetv3_large_100 p3_f2_role & p3_f3_role  (5.8 min)
pair 3/10 done: mobilenetv3_large_100 p3_f4_role & p3_f0_role  (9.8 min)
pair 4/10 done: efficientnet_b0 p3_f1_role & p3_f2_role  (13.8 min)
pair 5/10 done: efficientnet_b0 p3_f3_role & p3_f4_role  (17.9 min)
pair 6/10 done: mobilenetv4_conv_small p3_f0_role & p3_f1_role  (19.6 min)
pair 7/10 done: mobilenetv4_conv_small p3_f2_role & p3_f3_role  (21.3 min)
pair 8/10 done: mobilenetv4_conv_small p3_f4_role & p3_f0_role  (24.3 min)
pair 9/10 done: resnet50 p3_f1_role & p3_f2_role  (27.3 min)
pair 10/10 done: resnet50 p3_f3_role & p3_f4_role  (30.4 min)

=== PlantDoc P3 per fold ===
                                 model      proto  best_epoch  val_f1  test_f1  minutes
         mobilenetv3_large_100.ra_in1k p3_f1_role          25  0.6131   0.6247   2.6894
         mobilenetv3_large_100.ra_in1k p3_f0_role          30  0.6178   0.6402   2.7038
   

In [5]:
# ===== Step 6.4: PlantWild P2 baselines + P1 (official) runs for leakage comparison =====
import os, time, subprocess, pandas as pd
os.makedirs("/kaggle/working/students", exist_ok=True)
def run_pair(jobs):
    procs = []
    for gpu, args in enumerate(jobs):
        tag = "_".join(x for x in args if not x.startswith("--"))[:90]
        log = open(f"/kaggle/working/students/log_{tag}.txt", "w")
        procs.append((subprocess.Popen(["python", "/kaggle/working/train_student.py", *args],
                      env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}, stdout=log, stderr=subprocess.STDOUT), tag))
    for p, tag in procs:
        p.wait()
        if p.returncode != 0: print("FAILED", tag, open(f"/kaggle/working/students/log_{tag}.txt").read()[-2000:])

MB3, EB0 = "mobilenetv3_large_100.ra_in1k", "efficientnet_b0.ra_in1k"
MB4, R50 = "mobilenetv4_conv_small.e2400_r224_in1k", "resnet50.a1_in1k"
J = lambda d, pr, m, ep: ["--data", d, "--proto", pr, "--model", m, "--seed", "0", "--lr", "1e-3", "--epochs", str(ep)]
pairs = [[J("pw", "p2_role", EB0, 20), J("pw", "p2_role", R50, 20)],
         [J("pw", "p2_role", MB3, 20), J("pw", "p2_role", MB4, 20)],
         [J("pw", "p1_role", MB3, 20), J("pw", "p1_role", EB0, 20)],
         [J("pd", "p1_role", MB3, 30), J("pd", "p1_role", EB0, 30)]]
t0 = time.time()
for i, pr in enumerate(pairs):
    run_pair(pr)
    print(f"pair {i+1}/{len(pairs)} done: {[(j[1], j[3], j[5].split('.')[0]) for j in pr]}  ({(time.time()-t0)/60:.1f} min)")

R = pd.read_json("/kaggle/working/students/results.jsonl", lines=True)
R = R[R.get("pipeline") == "gpu_aug"].copy(); R["model"] = R.model.str.split(".").str[0]
R["proto"] = R.proto.str.replace(r"p3_f\d_role", "p3_role", regex=True)
T = R.groupby(["data", "proto", "model"]).agg(runs=("test_f1", "size"), params_M=("params_M", "first"),
        test_f1=("test_f1", "mean"), f1_sd=("test_f1", "std"), test_acc=("test_acc", "mean"),
        best_ep=("best_epoch", "mean"), min_per_run=("minutes", "mean")).round(4)
pd.set_option("display.width", 200)
print("\n=== ALL STUDENT BASELINES SO FAR ===\n", T.to_string())
T.to_csv("/kaggle/working/figures/table_students_baselines_all.csv")
print("\nStep 6.4 DONE.")

pair 1/4 done: [('pw', 'p2_role', 'efficientnet_b0'), ('pw', 'p2_role', 'resnet50')]  (13.0 min)
pair 2/4 done: [('pw', 'p2_role', 'mobilenetv3_large_100'), ('pw', 'p2_role', 'mobilenetv4_conv_small')]  (20.1 min)
pair 3/4 done: [('pw', 'p1_role', 'mobilenetv3_large_100'), ('pw', 'p1_role', 'efficientnet_b0')]  (32.2 min)
pair 4/4 done: [('pd', 'p1_role', 'mobilenetv3_large_100'), ('pd', 'p1_role', 'efficientnet_b0')]  (36.8 min)

=== ALL STUDENT BASELINES SO FAR ===
                                      runs  params_M  test_f1   f1_sd  test_acc  best_ep  min_per_run
data proto   model                                                                                  
pd   p1_role efficientnet_b0            1    4.0421   0.6200     NaN    0.6356     22.0       4.3645
             mobilenetv3_large_100      1    4.2366   0.5717     NaN    0.5890     12.0       2.9145
     p3_role efficientnet_b0            5    4.0421   0.6220  0.0324    0.6601     20.4       3.8750
             mobilenet

In [6]:
# ===== Step 6.5: student leakage + efficiency + Step 6 figures =====
import os, glob, time, json, numpy as np, pandas as pd, torch, timm, matplotlib.pyplot as plt
from torch.utils.flop_counter import FlopCounterMode
FEAT, STU, FIG = "/kaggle/working/feats", "/kaggle/working/students", "/kaggle/working/figures"
plt.rcParams.update({"font.size": 9, "savefig.bbox": "tight"})
LOG = f"{FIG}/figure_log.csv"; log = pd.read_csv(LOG).to_dict("records") if os.path.exists(LOG) else []
def save(fig, name, cap):
    for e in ["png", "pdf"]: fig.savefig(f"{FIG}/{name}.{e}", dpi=300)
    plt.close(fig); log[:] = [r for r in log if r["file"] != name] + [dict(file=name, step="6", caption=cap)]; print("saved", name)
M = {t: pd.read_csv(f"{FEAT}/{t}_manifest.csv", low_memory=False) for t in ["pd", "pw"]}
R = pd.read_json(f"{STU}/results.jsonl", lines=True); R = R[R.get("pipeline") == "gpu_aug"].copy()
R["short"] = R.model.str.split(".").str[0]
NICE = {"mobilenetv4_conv_small": "MobileNetV4-S", "mobilenetv3_large_100": "MobileNetV3-L",
        "efficientnet_b0": "EfficientNet-B0", "resnet50": "ResNet-50"}
ORDER = list(NICE)

# ---------- 1. student leakage inflation (P1 runs) ----------
rows = []
for _, r in R[R.proto == "p1_role"].iterrows():
    df = M[r.data]; L = np.load(f"{STU}/logits/{r.tag}_test.npy").astype(np.float32)
    idx = pd.read_csv(f"{STU}/logits/{r.tag}_test_uids.csv").row.values
    y = df.class_id.values[idx]; ok = L.argmax(1) == y
    twin = df.p1_test_twin.values[idx].astype(bool); conf = (df.drop_reason.values[idx] == "label_conflict")
    from sklearn.metrics import f1_score
    for sub, m in [("twin-in-train", twin & ~conf), ("twin-free", ~twin & ~conf)]:
        rows.append(dict(dataset=r.data, model=NICE[r.short], subset=sub, n=int(m.sum()), acc=ok[m].mean()))
    rows.append(dict(dataset=r.data, model=NICE[r.short], subset="official_f1_all", n=len(y), acc=f1_score(y, L.argmax(1), average="macro")))
    rows.append(dict(dataset=r.data, model=NICE[r.short], subset="official_f1_no_conflict", n=int((~conf).sum()),
                     acc=f1_score(y[~conf], L.argmax(1)[~conf], average="macro")))
T = pd.read_csv(f"{FEAT}/leakage_inflation.csv")
for _, t in T[T.subset != "all official"].iterrows():
    rows.append(dict(dataset=t.dataset, model="Teacher (DINOv2+CLIP)", subset=t.subset, n=t.n, acc=t.teacher_acc))
LK = pd.DataFrame(rows); LK.to_csv(f"{FIG}/table_student_leakage.csv", index=False)
piv = LK[LK.subset.isin(["twin-in-train", "twin-free"])].pivot_table(index=["dataset", "model"], columns="subset", values="acc")
piv["gap_pp"] = 100 * (piv["twin-in-train"] - piv["twin-free"])
print("=== LEAKAGE INFLATION (acc, conflicts excluded) ===\n", piv.round(4).to_string())
nz = LK[LK.subset.str.startswith("official")].pivot_table(index=["dataset", "model"], columns="subset", values="acc")
nz["noise_deflation_pp"] = 100 * (nz.official_f1_no_conflict - nz.official_f1_all)
print("\n=== NOISE DEFLATION on official test (macro-F1) ===\n", nz.round(4).to_string())

# ---------- 2. efficiency ----------
torch.set_num_threads(4); eff = []
for m in ["mobilenetv4_conv_small.e2400_r224_in1k", "mobilenetv3_large_100.ra_in1k", "efficientnet_b0.ra_in1k", "resnet50.a1_in1k"]:
    net = timm.create_model(m, pretrained=False, num_classes=89).eval(); x = torch.randn(1, 3, 224, 224)
    with FlopCounterMode(display=False) as fc: net(x)
    gmac = fc.get_total_flops() / 2e9
    with torch.no_grad():
        for _ in range(10): net(x)
        ts = [];  
        for _ in range(50): t = time.perf_counter(); net(x); ts.append(time.perf_counter() - t)
    cpu_ms = 1000 * np.median(ts)
    g = net.cuda().half(); xg = x.cuda().half()
    with torch.no_grad():
        for _ in range(20): g(xg)
        torch.cuda.synchronize(); ts = []
        for _ in range(100):
            t = time.perf_counter(); g(xg); torch.cuda.synchronize(); ts.append(time.perf_counter() - t)
    eff.append(dict(model=NICE[m.split(".")[0]], params_M=sum(p.numel() for p in net.parameters())/1e6,
                    GMACs=gmac, cpu_ms_bs1=cpu_ms, gpu_ms_bs1_fp16=1000*np.median(ts)))
    del net, g; torch.cuda.empty_cache()
E = pd.DataFrame(eff).round(3); E.to_csv(f"{FIG}/table_student_efficiency.csv", index=False)
print("\n=== EFFICIENCY (89-class head, 224x224) ===\n", E.to_string(index=False))

# ---------- 3. figures ----------
# fig6_1 learning curves
fig, axs = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for ax, (d, pr, title) in zip(axs, [("pd", "p3", "PlantDoc P3 (mean of 5 folds)"), ("pw", "p2", "PlantWild P2")]):
    for s in ORDER:
        fs = glob.glob(f"{STU}/curves/{d}_{pr}*_{s}_s0_lr0.001.csv") if pr == "p3" else glob.glob(f"{STU}/curves/{d}_{pr}_{s}_s0_lr0.001.csv")
        if not fs: continue
        C = pd.concat([pd.read_csv(f) for f in fs]).groupby("epoch").val_f1.mean()
        ax.plot(C.index, C.values, label=NICE[s])
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.grid(alpha=.3)
axs[0].set_ylabel("Validation macro-F1"); axs[1].legend(fontsize=8)
save(fig, "fig6_1_learning_curves", "Validation macro-F1 per epoch for the student baselines (fine-tuned from ImageNet).")

# fig6_2 baselines vs teacher
tp = pd.read_csv(f"{FIG}/table_teacher_probes.csv")
teach = {"pd": tp[(tp.dataset == "pd") & (tp.proto == "p3_role") & (tp.feat == "dino+clip")]["mean"].iloc[0],
         "pw": tp[(tp.dataset == "pw") & (tp.proto == "p2_role") & (tp.feat == "dino+clip")]["mean"].iloc[0]}
fig, axs = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for ax, (d, pr, title) in zip(axs, [("pd", "p3", "PlantDoc P3 (5-fold)"), ("pw", "p2_role", "PlantWild P2")]):
    sub = R[(R.data == d) & (R.proto.str.startswith(pr))]
    g = sub.groupby("short").test_f1.agg(["mean", "std"]).reindex(ORDER)
    ax.bar([NICE[s] for s in g.index], g["mean"], yerr=g["std"].fillna(0), capsize=3, color="#4C72B0")
    for i, v in enumerate(g["mean"]): ax.text(i, v + .01, f"{v:.3f}", ha="center", fontsize=8)
    ax.axhline(teach[d], color="#C44E52", ls="--", label=f"Teacher {teach[d]:.3f}")
    ax.set_title(title); ax.tick_params(axis="x", rotation=20); ax.set_ylim(0.4, 0.85); ax.legend(fontsize=8)
axs[0].set_ylabel("Test macro-F1")
save(fig, "fig6_2_baselines_vs_teacher", "Student baselines (test macro-F1) against the frozen DINOv2+CLIP teacher (dashed).")

# fig6_3 accuracy vs latency
fig, ax = plt.subplots(figsize=(5, 3.4))
for d, pr, mk in [("pd", "p3", "o"), ("pw", "p2_role", "s")]:
    g = R[(R.data == d) & (R.proto.str.startswith(pr))].groupby("short").test_f1.mean()
    for s, v in g.items():
        e = E[E.model == NICE[s]].iloc[0]
        ax.scatter(e.cpu_ms_bs1, v, marker=mk, s=25 + 3*e.params_M, color=f"C{ORDER.index(s)}")
        ax.annotate(NICE[s] if d == "pw" else "", (e.cpu_ms_bs1, v), fontsize=7, xytext=(4, 3), textcoords="offset points")
ax.scatter([], [], marker="o", color="gray", label="PlantDoc P3"); ax.scatter([], [], marker="s", color="gray", label="PlantWild P2")
ax.set_xlabel("CPU latency, batch 1 (ms)"); ax.set_ylabel("Test macro-F1"); ax.legend(fontsize=8); ax.grid(alpha=.3)
save(fig, "fig6_3_accuracy_vs_latency", "Accuracy-efficiency trade-off of student baselines (marker size ∝ parameters).")

# fig6_4 leakage: teacher vs students
fig, axs = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for ax, d, title in zip(axs, ["pd", "pw"], ["PlantDoc (official split)", "PlantWild (official split)"]):
    p = piv.loc[d]; x = np.arange(len(p))
    ax.bar(x - .2, p["twin-free"], .4, label="twin-free test", color="#4C72B0")
    ax.bar(x + .2, p["twin-in-train"], .4, label="twin-in-train test", color="#DD8452")
    for i, gp in enumerate(p.gap_pp): ax.text(i, 1.01, f"+{gp:.1f} pp", ha="center", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(p.index, rotation=15); ax.set_title(title); ax.set_ylim(0, 1.1)
axs[0].set_ylabel("Accuracy"); axs[0].legend(fontsize=8, loc="lower left")
save(fig, "fig6_4_leakage_students", "Leakage inflation for the teacher and fine-tuned students (label-conflict images excluded).")

pd.DataFrame(log).to_csv(LOG, index=False)
print("\nStep 6.5 DONE.")

=== LEAKAGE INFLATION (acc, conflicts excluded) ===
 subset                         twin-free  twin-in-train   gap_pp
dataset model                                                   
pd      EfficientNet-B0           0.6507         1.0000  34.9282
        MobileNetV3-L             0.5981         1.0000  40.1914
        Teacher (DINOv2+CLIP)     0.7943         1.0000  20.5700
pw      EfficientNet-B0           0.7098         0.9710  26.1137
        MobileNetV3-L             0.6915         0.9710  27.9409
        Teacher (DINOv2+CLIP)     0.7965         0.9585  16.2000

=== NOISE DEFLATION on official test (macro-F1) ===
 subset                   official_f1_all  official_f1_no_conflict  noise_deflation_pp
dataset model                                                                        
pd      EfficientNet-B0           0.6200                   0.6432              2.3200
        MobileNetV3-L             0.5717                   0.5971              2.5416
pw      EfficientNet-B0      

In [7]:
# ===== Step 7.1: protocol-specific cross-fitted teacher soft labels (for KD) =====
import numpy as np, pandas as pd, torch, torch.nn.functional as F, os, time
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
FEAT = "/kaggle/working/feats"; OUT = f"{FEAT}/kd_targets"; os.makedirs(OUT, exist_ok=True); dev = "cuda:0"
M = {t: pd.read_csv(f"{FEAT}/{t}_manifest.csv", low_memory=False) for t in ["pd", "pw"]}
R = pd.read_csv(f"{FEAT}/teacher_probe_results.csv")
WD = {t: R[(R.dataset == t) & (R.protocol == "p2_role") & (R.feat == "dino+clip")].wd.iloc[0] for t in M}
l2 = lambda a: a / np.linalg.norm(a, axis=1, keepdims=True)
def X_of(t):
    L = lambda k: np.load(f"{FEAT}/{t}_{k}.npy")
    return np.hstack([l2(np.hstack([L("dino_cls"), L("dino_mean")])), l2(L("clip"))]).astype(np.float32)
XF = {t: X_of(t) for t in M}

def fit_logits(Xtr, ytr, Xev, K, wd):
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    A = torch.tensor((Xtr - mu) / sd, device=dev); y = torch.tensor(ytr, device=dev)
    W = torch.zeros(A.shape[1], K, device=dev, requires_grad=True); b = torch.zeros(K, device=dev, requires_grad=True)
    opt = torch.optim.LBFGS([W, b], lr=1, max_iter=300, line_search_fn="strong_wolfe")
    def c():
        opt.zero_grad(); l = F.cross_entropy(A @ W + b, y) + wd * (W**2).sum(); l.backward(); return l
    opt.step(c)
    return (torch.tensor((Xev - mu) / sd, device=dev) @ W.detach() + b.detach()).cpu().numpy()

def fit_temperature(logits, y):
    z = torch.tensor(logits, device=dev); yy = torch.tensor(y, device=dev)
    lt = torch.zeros(1, device=dev, requires_grad=True); opt = torch.optim.LBFGS([lt], lr=.1, max_iter=100)
    def c():
        opt.zero_grad(); l = F.cross_entropy(z / lt.exp(), yy); l.backward(); return l
    opt.step(c); return float(lt.exp())

PROTOS = [("pd", f"p3_f{k}_role") for k in range(5)] + [("pd", "p2_role"), ("pd", "p1_role"),
          ("pw", "p2_role"), ("pw", "p1_role")]
rows, t0 = [], time.time()
for tag, pc in PROTOS:
    df = M[tag]; X = XF[tag]; y = df.class_id.values; K = int(y.max()) + 1; r = df[pc].values
    tr = np.where(r == "train")[0]; ev = np.where(np.isin(r, ["val", "test"]))[0]
    g = df.dup_group.astype(str).where(df.dup_group.notna(), "solo_" + df.uid.astype(str)).values
    Z = np.zeros((len(df), K), np.float32)
    for a_, b_ in StratifiedGroupKFold(5, shuffle=True, random_state=42).split(X[tr], y[tr], g[tr]):
        Z[tr[b_]] = fit_logits(X[tr[a_]], y[tr[a_]], X[tr[b_]], K, WD[tag])          # cross-fitted train logits
    Z[ev] = fit_logits(X[tr], y[tr], X[ev], K, WD[tag])                               # full-train probe for val/test
    va = np.where(r == "val")[0]; T = fit_temperature(Z[va], y[va])
    np.save(f"{OUT}/{tag}_{pc.replace('_role','')}_logits.npy", Z.astype(np.float16))
    P = torch.softmax(torch.tensor(Z[tr]), 1).numpy()
    te = np.where(r == "test")[0]
    rows.append(dict(dataset=tag, protocol=pc, n_train=len(tr), T_val=round(T, 3),
                     trainCF_acc=(P.argmax(1) == y[tr]).mean(), trainCF_disagree=(P.argmax(1) != y[tr]).mean(),
                     trainCF_mean_maxp=P.max(1).mean(), trainCF_p_given=P[np.arange(len(tr)), y[tr]].mean(),
                     test_f1=f1_score(y[te], Z[te].argmax(1), average="macro")))
    print(f"done {tag} {pc} ({time.time()-t0:.0f}s)")
S = pd.DataFrame(rows).round(4); pd.set_option("display.width", 200)
print("\n=== KD TARGETS (cross-fitted within protocol train split) ===\n", S.to_string(index=False))
S.to_csv(f"{FEAT}/kd_targets/summary.csv", index=False)
print("\nStep 7.1 DONE.")

/tmp/ipykernel_58/3902930053.py:30: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  opt.step(c); return float(lt.exp())


done pd p3_f0_role (8s)
done pd p3_f1_role (15s)
done pd p3_f2_role (23s)
done pd p3_f3_role (32s)
done pd p3_f4_role (41s)
done pd p2_role (51s)
done pd p1_role (62s)
done pw p2_role (90s)
done pw p1_role (119s)

=== KD TARGETS (cross-fitted within protocol train split) ===
 dataset   protocol  n_train  T_val  trainCF_acc  trainCF_disagree  trainCF_mean_maxp  trainCF_p_given  test_f1
     pd p3_f0_role     1648  1.133       0.7791            0.2209             0.8061           0.7049   0.7822
     pd p3_f1_role     1648  1.115       0.7791            0.2209             0.8017           0.6979   0.7847
     pd p3_f2_role     1648  1.092       0.7809            0.2191             0.8043           0.7043   0.7624
     pd p3_f3_role     1648  1.187       0.7743            0.2257             0.7975           0.6990   0.8010
     pd p3_f4_role     1649  0.979       0.7835            0.2165             0.7983           0.7035   0.7620
     pd    p2_role     1872  1.111       0.7751          

In [8]:
import os
os.system("cd /kaggle/working && zip -qr wildleaf_step6.zip students figures feats train_student.py")
print(os.path.getsize("/kaggle/working/wildleaf_step6.zip")/1e9, "GB")

1.113574803 GB
